In [ ]:
!pip install -q datasets

In [ ]:
import os, json, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset, load_from_disk, DatasetDict
from sklearn.model_selection import train_test_split, GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- outputs (persist only after Save Version / New Dataset) ----
PROJECT_DIR = "/kaggle/working/AI_Image_Generalization"
META_DIR    = f"{PROJECT_DIR}/metadata"
SPLITS_DIR  = f"{PROJECT_DIR}/splits"
RESULTS_DIR = f"{PROJECT_DIR}/results"
CKPT_DIR    = f"{PROJECT_DIR}/checkpoints"
for d in [META_DIR, SPLITS_DIR, RESULTS_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# HF download cache goes to scratch so it doesn't eat the 20GB working limit
HF_CACHE = "/kaggle/temp/hf_cache" if os.path.isdir("/kaggle/temp") else "/tmp/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
IMG_COL, CAP_COL = "Image", "Caption"      # verify in Cell 5

def find_dirs(top, names, max_depth=5):
    """Find folders by name under `top` without walking into the big image folders."""
    found, base = [], top.rstrip(os.sep).count(os.sep)
    for root, dirs, _ in os.walk(top):
        if root.count(os.sep) - base >= max_depth:
            dirs[:] = []
            continue
        for d in list(dirs):
            if d.lower() in names:
                found.append(os.path.join(root, d))
            if d.lower() in {"ai", "nature"}:
                dirs.remove(d)                 # never descend into image folders
    return found

# ---- mode: is my saved dataset attached? ----
saved = find_dirs("/kaggle/input", {"defactify_subset"}, max_depth=4)
RESUME = len(saved) > 0
SAVED_ROOT = os.path.dirname(saved[0]) if RESUME else None

print("MODE:", "RESUME (loading from saved dataset)" if RESUME else "FIRST RUN (building from scratch)")
if RESUME:
    print("Saved root:", SAVED_ROOT)

In [ ]:
print("Inputs:", os.listdir("/kaggle/input"))
found = find_dirs("/kaggle/input", {"ai", "nature"}, max_depth=8)
print("Found:", *found, sep="\n  ")
assert found, "BigGAN input folders not found in /kaggle/input"

BIGGAN_DIR = os.path.commonpath(found)
while os.path.basename(BIGGAN_DIR).lower() in {"ai", "nature", "train", "val"}:
    BIGGAN_DIR = os.path.dirname(BIGGAN_DIR)
print("\nBIGGAN_DIR:", BIGGAN_DIR, "\n")


In [ ]:
import time

if RESUME:
    print("RESUME mode: skipping, metadata already saved.")
else:
    t0 = time.time()
    rows = []
    for folder in sorted(found):
        rel_folder = os.path.relpath(folder, BIGGAN_DIR)
        parts = [x.lower() for x in Path(rel_folder).parts]
        is_ai = 1 if parts[-1] == "ai" else 0
        off = "train" if "train" in parts else "val"

        n = 0
        for root, _, files in os.walk(folder):
            sub = os.path.relpath(root, BIGGAN_DIR)
            for f in files:
                if os.path.splitext(f)[1].lower() in IMG_EXT:
                    rel = os.path.join(sub, f)
                    rows.append((
                        "bg_" + rel, "local", rel, None, -1,
                        "GenImage_BigGAN" if is_ai else "ImageNet_real",
                        "BigGAN" if is_ai else "real",
                        is_ai, "bg_" + rel, off,
                    ))
                    n += 1
        print(f"{rel_folder:25s} {n:>8,} files   ({time.time()-t0:.0f}s elapsed)")

    bg = pd.DataFrame(rows, columns=[
        "uid", "storage", "image_path", "hf_split", "hf_index",
        "source_dataset", "generator", "binary_label", "group", "official_split"])
    print("\n", bg.groupby(["official_split", "generator"]).size())

    bg["split"] = "test"                       # official val -> our test
    m = bg.official_split == "train"           # official train -> 90/10 train/val
    tr_idx, va_idx = train_test_split(bg[m].index, test_size=0.10,
                                      random_state=SEED, stratify=bg[m]["binary_label"])
    bg.loc[tr_idx, "split"] = "train"
    bg.loc[va_idx, "split"] = "val"
    bg = bg.drop(columns="official_split")

    bg.to_csv(f"{META_DIR}/biggan_metadata.csv", index=False)
    print("\nBigGAN final:")
    print(bg.groupby(["split", "generator"]).size())
    print(f"Done in {time.time()-t0:.0f}s")

In [ ]:
if RESUME:
    defactify = load_from_disk(os.path.join(SAVED_ROOT, "defactify_subset"))
    bal = pd.read_csv(os.path.join(SAVED_ROOT, "metadata", "balanced_metadata.csv"))
    SPLITS_DIR = os.path.join(SAVED_ROOT, "splits")          # read-only saved splits
    print("Loaded saved Defactify subset + balanced metadata + splits.")
    print(bal.groupby(["split", "generator"]).size().unstack(fill_value=0))
    print("\nExperiments:", sorted(os.listdir(SPLITS_DIR)))
    print("\n>>> RESUME: jump straight to Cell 13 (loader). Skip Cells 6-12.")
else:
    defactify = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset", cache_dir=HF_CACHE)
    print("Loaded from HF (one-time).")

print(defactify)
cols = defactify[list(defactify.keys())[0]].column_names
print(cols)
assert IMG_COL in cols, f"Fix IMG_COL, columns are {cols}"
print("Has caption column:", CAP_COL in cols)

In [ ]:
GEN_MAP = {
    0: "real",
    1: "Stable_Diffusion_2_1",
    2: "Stable_Diffusion_XL",
    3: "Stable_Diffusion_3",
    4: "DALL_E_3",
    5: "Midjourney_v6",
}

parts = []
for hf_split in defactify:
    ds = defactify[hf_split]
    la = np.array(list(ds["Label_A"]))
    lb = np.array(list(ds["Label_B"]))
    if CAP_COL in ds.column_names:
        cap = [str(c).strip().lower() for c in ds[CAP_COL]]
    else:
        cap = [f"{hf_split}_{i}" for i in range(len(ds))]   # no grouping possible

    parts.append(pd.DataFrame({
        "uid": [f"df_{hf_split}_{i}" for i in range(len(ds))],
        "storage": "hf",
        "image_path": None,
        "hf_split": hf_split,
        "hf_index": np.arange(len(ds)),
        "binary_label": la,
        "label_b": lb,
        "group": ["cap_" + c for c in cap],
    }))

df_def = pd.concat(parts, ignore_index=True)

# sanity: Label_A and Label_B must agree on real vs AI
print(pd.crosstab(df_def["binary_label"], df_def["label_b"]))
assert ((df_def.binary_label == 0) == (df_def.label_b == 0)).all(), "Label_A / Label_B mismatch"

df_def["generator"] = df_def["label_b"].map(GEN_MAP)
df_def["source_dataset"] = np.where(df_def.binary_label == 0, "Defactify_real", "Defactify")
df_def = df_def.drop(columns="label_b")

print(df_def.generator.value_counts())

In [ ]:
g1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
trv_i, te_i = next(g1.split(df_def, groups=df_def["group"]))

trv = df_def.iloc[trv_i]
g2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=SEED)
tr_i, va_i = next(g2.split(trv, groups=trv["group"]))

df_def["split"] = ""
df_def.loc[trv.index[tr_i], "split"] = "train"
df_def.loc[trv.index[va_i], "split"] = "val"
df_def.loc[df_def.index[te_i], "split"] = "test"

# leakage check: no caption group in >1 split
g = df_def.groupby("group")["split"].nunique()
print("Groups spanning multiple splits:", (g > 1).sum())
print(df_def.groupby(["split", "generator"]).size().unstack(fill_value=0))

In [ ]:
COLS = ["uid", "storage", "image_path", "hf_split", "hf_index",
        "source_dataset", "generator", "binary_label", "group", "split"]

full = pd.concat([bg[COLS], df_def[COLS]], ignore_index=True)
assert full.uid.is_unique

full.to_csv(f"{META_DIR}/full_metadata.csv", index=False)
print(len(full))
print(full.groupby(["split", "generator"]).size().unstack(fill_value=0))

In [ ]:
CAPS = {"train": 8000, "val": 1000, "test": 1000}   # change once you see the counts

full["bucket"] = np.where(full.binary_label == 0, full.source_dataset, full.generator)

parts = []
for (split, bucket), d in full.groupby(["split", "bucket"]):
    n = min(CAPS[split], len(d))
    parts.append(d.sample(n=n, random_state=SEED))

bal = pd.concat(parts, ignore_index=True).drop(columns="bucket")
bal.to_csv(f"{META_DIR}/balanced_metadata.csv", index=False)

print(len(bal))
print(bal.groupby(["split", "generator"]).size().unstack(fill_value=0))
print(bal.groupby(["split", "source_dataset"]).size().unstack(fill_value=0))

In [ ]:
flag = f"{PROJECT_DIR}/.subset_saved"
assert not os.path.exists(flag), "Already ran. Re-running would corrupt hf_index."

sub = {}
for sp in defactify:
    m = (bal.storage == "hf") & (bal.hf_split == sp)
    idx = np.sort(bal.loc[m, "hf_index"].astype(int).unique())
    sub[sp] = defactify[sp].select(idx)
    bal.loc[m, "hf_index"] = np.searchsorted(idx, bal.loc[m, "hf_index"].astype(int))
DatasetDict(sub).save_to_disk(f"{PROJECT_DIR}/defactify_subset")

bal.to_csv(f"{META_DIR}/balanced_metadata.csv", index=False)
open(flag, "w").write("done")
print("Saved. Now run Cells 10-12 so the split CSVs use the remapped indices.")

In [ ]:
AI_GENS = sorted(bal[bal.binary_label == 1].generator.unique())
GAN_GENS = ["BigGAN"]
DIFF_GENS = [g for g in AI_GENS if g not in GAN_GENS]
print("AI generators:", AI_GENS)

REAL_FOR = {"GenImage_BigGAN": "ImageNet_real", "Defactify": "Defactify_real"}

def matched_real_test(ai_test):
    """Real test images from the same source family as the AI test images, equal count, test split only."""
    out = []
    for ai_src, real_src in REAL_FOR.items():
        n = (ai_test.source_dataset == ai_src).sum()
        if n == 0:
            continue
        pool = bal[(bal.split == "test") & (bal.source_dataset == real_src)]
        out.append(pool.sample(n=min(n, len(pool)), random_state=SEED))
    return pd.concat(out)

def build_experiment(name, train_gens, test_gens):
    def part(split, gens):
        ai = bal[(bal.split == split) & (bal.binary_label == 1) & (bal.generator.isin(gens))]
        return ai

    # train / val: AI from train_gens + all real of that split
    tr = pd.concat([part("train", train_gens), bal[(bal.split == "train") & (bal.binary_label == 0)]])
    va = pd.concat([part("val", train_gens),   bal[(bal.split == "val")   & (bal.binary_label == 0)]])

    # test: AI from test_gens + matched real
    te_ai = part("test", test_gens)
    te = pd.concat([te_ai, matched_real_test(te_ai)])

    # leakage checks
    assert not (set(tr.uid) & set(va.uid)) and not (set(tr.uid) & set(te.uid)) and not (set(va.uid) & set(te.uid))
    assert not (set(tr.group) & set(te.group) & set(df_def.group)), "caption group leak"
    assert set(test_gens).isdisjoint(set(train_gens)) or name == "seen_all"

    out = f"{SPLITS_DIR}/{name}"
    os.makedirs(out, exist_ok=True)
    for sp, d in [("train", tr), ("val", va), ("test", te)]:
        d.sample(frac=1, random_state=SEED).to_csv(f"{out}/{sp}.csv", index=False)
    print(f"{name:35s} train={len(tr):6d}  val={len(va):5d}  test={len(te):5d}  "
          f"(test AI={int((te.binary_label==1).sum())}, real={int((te.binary_label==0).sum())})")

In [ ]:
# 1) seen generators (in-distribution baseline)
build_experiment("seen_all", AI_GENS, AI_GENS)

# 2) leave-one-generator-out
for g in AI_GENS:
    build_experiment(f"unseen_{g}", [x for x in AI_GENS if x != g], [g])

# 3) GAN -> Diffusion, Diffusion -> GAN
build_experiment("gan_to_diffusion", GAN_GENS, DIFF_GENS)
build_experiment("diffusion_to_gan", DIFF_GENS, GAN_GENS)

In [ ]:
gen2id = {g: i for i, g in enumerate(AI_GENS)}
json.dump(gen2id, open(f"{META_DIR}/generator_to_id.json", "w"), indent=2)

os.makedirs(f"{SPLITS_DIR}/attribution", exist_ok=True)
for sp in ["train", "val", "test"]:
    d = bal[(bal.split == sp) & (bal.binary_label == 1)].copy()
    d["generator_id"] = d.generator.map(gen2id)
    d.to_csv(f"{SPLITS_DIR}/attribution/{sp}.csv", index=False)
    print(sp, len(d))
print(gen2id)

In [ ]:
import io, torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

class UnifiedImageDataset(Dataset):
    def __init__(self, csv_path, hf_data, local_root, transform=None, label_col="binary_label"):
        self.rows = pd.read_csv(csv_path).to_dict("records")
        self.hf, self.root, self.tf, self.label_col = hf_data, local_root, transform, label_col
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        if r["storage"] == "hf":
            img = self.hf[r["hf_split"]][int(r["hf_index"])][IMG_COL]
        else:
            img = Image.open(os.path.join(self.root, r["image_path"]))
        img = img.convert("RGB")
        if self.tf: img = self.tf(img)
        return img, int(r[self.label_col])

class JPEGRecompress:
    """Round-trip every image (including BigGAN PNGs) through JPEG so format isn't a cue."""
    def __init__(self, q_min=65, q_max=95):
        self.q_min, self.q_max = q_min, q_max
    def __call__(self, img):
        q = random.randint(self.q_min, self.q_max)
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

CROP = 128
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomCrop(CROP, pad_if_needed=True),   # crop first: JPEG on a 1024px image is slow
    transforms.RandomHorizontalFlip(),
    JPEGRecompress(65, 95),
    transforms.ToTensor(), norm,
])
eval_tf = transforms.Compose([
    transforms.CenterCrop(CROP),                        # pads if an image is smaller than 128
    JPEGRecompress(90, 90),
    transforms.ToTensor(), norm,
])


In [ ]:
for exp in sorted(os.listdir(SPLITS_DIR)):
    csv = f"{SPLITS_DIR}/{exp}/train.csv"
    if not os.path.exists(csv):
        continue
    ds = UnifiedImageDataset(csv, defactify, BIGGAN_DIR, transform=eval_tf)
    x, y = next(iter(DataLoader(ds, batch_size=32, shuffle=True, num_workers=2)))
    print(f"{exp:32s} batch {tuple(x.shape)}  label mean={y.float().mean():.2f}")

# explicitly confirm BOTH storage types decode
d = pd.read_csv(f"{SPLITS_DIR}/gan_to_diffusion/train.csv")
ds = UnifiedImageDataset(f"{SPLITS_DIR}/gan_to_diffusion/train.csv", defactify, BIGGAN_DIR)
for s in ["local", "hf"]:
    i = d.index[d.storage == s][0]
    img, lab = ds[i]
    print(s, "->", img.size, "label", lab, "|", d.loc[i, "generator"])

In [ ]:
print("Experiments:", sorted(os.listdir(SPLITS_DIR)))
print("\nBalanced counts (split x label):")
print(bal.groupby(["split", "binary_label"]).size().unstack(fill_value=0))
print("\nPer-source:")
print(bal.groupby(["split", "source_dataset"]).size().unstack(fill_value=0))

!du -sh /kaggle/working/AI_Image_Generalization/* 2>/dev/null

In [ ]:
d = pd.read_csv(f"{SPLITS_DIR}/seen_all/train.csv")
recs = []
for (src, gen), g in d.groupby(["source_dataset", "generator"]):
    for _, r in g.sample(min(150, len(g)), random_state=0).iterrows():
        if r.storage == "hf":
            im = defactify[r.hf_split][int(r.hf_index)][IMG_COL]
        else:
            im = Image.open(os.path.join(BIGGAN_DIR, r.image_path))
        recs.append(dict(source=src, generator=gen, w=im.size[0], h=im.size[1], fmt=im.format))
s = pd.DataFrame(recs)

print(s.groupby(["source", "generator"]).agg(
    w_med=("w", "median"), h_med=("h", "median"), w_min=("w", "min"), w_max=("w", "max")))
print()
print(pd.crosstab([s.source, s.generator], s.fmt))